## 0. 準備環境（直接執行，不必逐行看懂）
先依序執行下面兩格，安裝所需套件。環境已可用時會自動跳過。

**Colab 請分開執行。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格安裝 PyGMT 等套件。勿在安裝期間重複按執行。

此教材使用 PyGMT 0.17 / GMT 6.5。新的 Colab 執行環境仍需安裝。


In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 1/2：安裝 Conda（約 1 分鐘）。完成後 Colab 會自動重啟執行環境，等重新連線再執行下一格。", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")


In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 2/2：安裝 PyGMT 與相依套件（約 2–4 分鐘），下方會逐行顯示進度。", flush=True)
    command = ["mamba", "install", "-y", "-c", "conda-forge", "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas"]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
    if process.returncode != 0:
        raise RuntimeError(f"安裝失敗（exit {process.returncode}），請重新執行本格或重啟執行環境。")
    print("安裝完成，可以往下執行。")
else:
    print("跳過 Colab 安裝。")


# 01｜基本地圖與地震

從空白底圖開始，加上海岸線與地震，再用大小和顏色表達規模與深度。先自己改參數，暫時不用 AI。

請先另存副本，完成環境設置後，由上往下執行。

[課前介紹](https://github.com/jimmy60504/pygmt-map-lab/blob/main/intro.md) · [課程首頁](https://github.com/jimmy60504/pygmt-map-lab)

[1｜基本地圖與地震](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/01_maps_earthquakes.ipynb) · [2｜地形與 3D](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/02_terrain_3d.ipynb) · [3｜AI 探索與作業](https://github.com/jimmy60504/pygmt-map-lab/blob/main/03_ai_exploration.md)


### 常用快捷鍵

| 操作 | Windows／Linux | Mac |
| --- | --- | --- |
| 執行目前儲存格並移到下一格 | Shift + Enter | Shift + Enter |
| 取消／切換註解 | Ctrl + / | ⌘ + / |

把游標放在程式行，或選取多行，再按切換註解的快捷鍵，即可移除或加上行首的 `#`。只選程式行，不要連中文說明一起取消註解。

修改後按 **Shift + Enter** 看結果，等執行完成再繼續下一步。


## 1. 從空白底圖，一步一步畫出台灣
先執行下一格，看到最簡單的地圖外框。接著依序移除步驟 1–6 程式行開頭的 `#`，每次只開啟一步，保留前面已開啟的步驟，再重跑整格觀察差異。

- `fig = pygmt.Figure()`：建立一張新圖；每次重跑都從頭畫，不會累積上次的內容。
- `region`：繪圖範圍，順序是 **西、東、南、北**；`projection="M15c"`：麥卡托投影、圖寬 15 公分。
- `fig.basemap()` 畫框線、刻度等；`fig.coast()` 畫海陸與海岸線；最後 `fig.show()` 顯示結果。

**先記住繪圖順序**：在同一個 `fig` 上，後畫的內容可能蓋住前面的內容。因此先填海陸顏色，再加線條、格線與標題。

後面的呼叫省略 `region` 與 `projection`，是沿用這張圖已設定的範圍與投影，**不是讀取上一層的圖片**。圖層疊加與設定沿用是兩件事。

### 第一張圖的 API 參考
不用整頁讀完：想改哪個效果，就點對應指令，在 **Parameters** 找參數，再看 **Examples**。以下連結對應課堂使用的 PyGMT 0.17。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [pygmt.Figure()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.html) | 建立一張圖 | Methods：還能加哪些內容 |
| [fig.basemap()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.basemap.html) | 底圖、刻度、格線與標題 | `region`、`projection`、`frame` |
| [fig.coast()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.coast.html) | 海陸填色與海岸線 | `land`、`water`、`shorelines`、`resolution` |
| [fig.show()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.show.html) | 顯示目前的圖 | `width`、`dpi` |

`import pygmt` 是載入套件，不是繪圖指令。`region=[119, 123, 21, 26]` 依序指定西界、東界、南界、北界。

**試著發現一個新選項**：打開 `coast` 文件，找找 `borders` 或 `rivers` 能做什麼。


In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame="0")

# 1. 加上經緯度數字與刻度（a：標示數字；f：刻度）
# fig.basemap(frame="af")

# 2. 填上陸地與海洋顏色
# fig.coast(land="gray90", water="lightblue", resolution="h")

# 3. 加上海岸線
# fig.coast(shorelines="0.6p,gray30", resolution="h")

# 4. 加上每 1 度的經緯格線，並重畫刻度（g：格線）
# fig.basemap(frame="a1f0.5g1")

# 5. 加上標題
# fig.basemap(frame="+tTaiwan: coastlines")

# 6. 加上 50 km 比例尺
# fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")

fig.show()
# 圖片顯示後，可按右鍵另存圖片。


## 2. 地震在哪裡？

還不熟悉地震學的圖長什麼樣？先翻 [地震學常見圖像](https://github.com/jimmy60504/pygmt-map-lab/blob/main/earthquake-figure-guide.md)：每種圖想回答什麼、怎麼讀，並附範例圖。這節要畫的就是其中最基本的地震分布圖。

### 先挑一個你想畫的地震事件

不知道要選哪個事件？台灣事件可以先到氣象署看看：

- [最近地震](https://www.cwa.gov.tw/V8/C/E/index.html)：從近期有感地震找題目，點進報告看發生時間、位置與規模。
- [歷史災害地震](https://scweb.cwa.gov.tw/zh-tw/page/disaster)：認識集集、美濃等重大事件，找一個你想進一步觀察的地震。較早期的歷史記載不一定有完整的 USGS 資料，練習可先選近代事件。

想畫世界其他地方，則可以從 USGS 找題目：

- [Latest Earthquakes｜最近地震地圖](https://earthquake.usgs.gov/earthquakes/map/)：瀏覽近期全球地震，可選「30 Days, Significant Worldwide」找最近一個月的重要事件。
- [Significant Earthquakes｜重大地震](https://earthquake.usgs.gov/earthquakes/browse/significant.php)：依年份找事件。「重大」不只看規模，也考慮有感回報與可能影響。
- [Search Earthquake Catalog｜地震目錄查詢](https://earthquake.usgs.gov/earthquakes/search/)：設定時間、規模與區域，也可以選 CSV 輸出。

選好後，記下時間與震央位置，把下方網址改成事件前後幾天、震央附近的範圍，觀察主震周邊的地震分布；地圖的 `region` 也要配合調整。

**時間要對齊**：氣象署報告使用台灣時間（UTC+8），下方 USGS 查詢使用 UTC，要先減 8 小時，日期也可能變成前一天。這裡用氣象署找題目，實際繪圖資料仍來自 USGS；兩邊的規模與位置可能不同，不必強求完全一致。


### 用 USGS API 取得資料

向 USGS 要一份地震資料，再把經緯度畫到地圖上。API 就像點餐：在網址指定時間、區域與最低規模，USGS 就回傳符合條件的 CSV 表格。

這次查詢 **2020-01-01 至執行當下（UTC）、規模 ≥ 2**，範圍是東經 119–123 度、北緯 21–26 度，與前面的地圖相同。觀察地震位置與深度的空間變化；USGS 對台灣小地震的收錄不完整，這些點不代表所有台灣地震，也不是完整的隱沒板塊形狀。這組圖呈現多年地震分布；第一張另外標記 2024 花蓮主震來示範符號，不代表其他地震都是它的餘震。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 按條件取得地震 CSV | `format`、`starttime`、`endtime`、`minmagnitude`、經緯度範圍 |
| [pd.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) | 把 CSV 網址讀成表格 | `filepath_or_buffer`：此處傳入 `url` |

可以複製下格印出的網址到瀏覽器下載資料；每次執行都需要網路。


In [ ]:
import pandas as pd
from datetime import datetime, timezone


# 1. 設定結束時間：每次執行當下的 UTC 時間
endtime = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")


# 2. 組合查詢網址：修改下方時間、規模與範圍
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"  # USGS 地震查詢入口
    "?format=csv"                    # 回傳 CSV 表格；第一個參數用 ?
    "&starttime=2020-01-01"           # 開始時間（UTC）；後續參數用 &
    f"&endtime={endtime}"             # 結束時間（UTC）
    "&minmagnitude=2"                 # 最低地震規模
    "&minlongitude=119"               # 西界：最小經度
    "&maxlongitude=123"               # 東界：最大經度
    "&minlatitude=21"                 # 南界：最小緯度
    "&maxlatitude=26"                 # 北界：最大緯度
    "&orderby=time-asc"              # 依時間由早到晚排列
)


# 3. 讀取資料，去掉缺少繪圖欄位的地震
quakes = pd.read_csv(url)
quakes = quakes.dropna(subset=["longitude", "latitude", "mag", "depth"])


# 4. 確認查詢網址、筆數與前五筆資料
print(url)  # 可複製到瀏覽器，查看同一份 CSV
print(f"可繪製的地震：{len(quakes)} 筆；深度單位：km；時間：UTC。")

quakes[["time", "longitude", "latitude", "mag", "depth"]].head()


### 第一張：從 Pandas 表格取經緯度畫點
`quakes` 是 Pandas 的 DataFrame：每列是一筆地震，欄位包含經度、緯度、規模與深度。

這裡不是用 Pandas 畫圖，而是把 `quakes.longitude` 與 `quakes.latitude` 交給 PyGMT 的 `fig.plot()`。先讓所有點一樣大、同一種顏色，只看地震在哪裡。

**圓圈之外，也能畫星星**：`style="c0.12c"` 是直徑 0.12 cm 的圓圈，`style="a0.6c"` 是大小 0.6 cm 的星形。下格用星星標出 [2024 花蓮主震（USGS）](https://earthquake.usgs.gov/earthquakes/eventpage/us7000m9g4/executive)，示範直接指定經緯度。星星最後畫，會疊在圓圈上。

**試看看**：將星星的 `a` 改成 `t`（三角形）或 `s`（正方形），保持 `0.6c` 不變。每張圖都重新建立 `fig`，不會疊到上一張；第二、三張先不加這個標記。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 畫點與不同符號 | `x`、`y`、`style`、`fill`、`pen`、`label` |
| [fig.legend()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.legend.html) | 顯示符號圖例 | `position`、`box` |


In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")
fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")  # 50 km 比例尺，參考緯度 24°N


# 從 Pandas 表格取出經緯度，每一列畫成一個點
fig.plot(
    x=quakes.longitude,  # 經度欄位：每筆地震的 x 位置
    y=quakes.latitude,  # 緯度欄位：每筆地震的 y 位置
    style="c0.12c",  # 所有圓圈的直徑都一樣：0.12 cm
    label="Catalog events (M >= 2)",
    fill="tomato",  # 固定顏色：所有地震都用同一種色
    pen="0.25p,gray20",
    transparency=40,
)


# 單獨標出 2024 花蓮主震（USGS：us7000m9g4）
fig.plot(
    x=121.5976,       # 經度
    y=23.8356,        # 緯度
    style="a0.6c",    # a：星形；t：三角形；s：正方形
    fill="yellow",
    pen="1p,black",
    label="2024 Hualien mainshock",  # 新增：把星星與文字放進圖例
)

fig.legend(position="JTL+jTL+o0.2c", box="+gwhite+p0.5p")


fig.show()


### 第二張：讓地震規模決定大小
把固定大小的 `style="c0.12c"` 改成 `style="c"`，另外用 `size` 傳入每筆地震的圓圈直徑；顏色先維持不變。

用 `if / elif / else` 分區間，同一區間用固定直徑：M<3 → 0.035 cm、3≤M<4 → 0.07 cm、4≤M<5 → 0.14 cm、5≤M<6 → 0.28 cm、6≤M<7 → 0.56 cm、M≥7 → 1.12 cm。`plot_quakes.mag.apply(magnitude_size)` 會把每筆規模交給這個函式判斷。這是視覺設計，不是能量比例或影響半徑；規模也不是各地感受到的震度。

**試看看**：`ascending=True` 讓小圓先畫、大圓後畫；改成 `False`，比較重疊效果。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 讓每筆地震有不同大小 | `size`、`style`、`transparency` |
| [DataFrame.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) | 決定資料與繪圖順序 | `by`：此處為 `"mag"`；`ascending` |


In [ ]:
import pygmt
from io import StringIO

# 依規模區間指定圓圈直徑（cm）；同一區間使用相同大小
def magnitude_size(magnitude):
    if magnitude < 3:
        return 0.035
    elif magnitude < 4:
        return 0.07
    elif magnitude < 5:
        return 0.14
    elif magnitude < 6:
        return 0.28
    elif magnitude < 7:
        return 0.56
    else:
        return 1.12


# 小圓先畫，大圓後畫
plot_quakes = quakes.sort_values("mag", ascending=True)


fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")
fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")  # 50 km 比例尺，參考緯度 24°N


# 改用 size，讓每筆地震有自己的圓圈大小
fig.plot(
    x=plot_quakes.longitude,
    y=plot_quakes.latitude,
    style="c",  # 改動：只指定圓形，大小改由下一行決定
    size=plot_quakes.mag.apply(magnitude_size),  # 新增：把每筆規模換成圓圈直徑
    fill="tomato",  # 沿用：這張先不改顏色
    pen="0.25p,gray20",
    transparency=40,
)


# 加上規模大小圖例
legend = StringIO(
    "".join(
        f"S 0.6c c {magnitude_size(mag):.3f}c gray70 0.25p,gray20 1.4c {label}\n"
        f"G {max(0.15, magnitude_size(mag) - 0.25):.2f}c\n"
        for mag, label in [
            (2, "M < 3"),
            (3, "3 <= M < 4"),
            (4, "4 <= M < 5"),
            (5, "5 <= M < 6"),
            (6, "6 <= M < 7"),
            (7, "M >= 7"),
        ]
    )
)

fig.legend(
    spec=legend,
    position="JTL+jTL+o0.2c",
    box="+gwhite+p0.5p",
)


fig.show()


### 第三張：用深度上色，認識 CPT

前一張用規模控制大小；這張保留大小，再用 `fill=plot_quakes.depth` 與 `cmap=True` 把深度轉成顏色，最後加上 color bar。

CPT（Color Palette Table）決定數值對應什麼顏色。從下方色票總覽挑一組喜歡的配色，把名稱填進 `pygmt.makecpt(cmap=...)`。

- 這次用 `cmap="batlow"`：以感知較均勻、色覺友善的循序色票呈現深度；以色條讀取數值。
- 可以換成之前的 `gmt/seis` 或 `rainbow` 比較：熟悉的配色不一定對所有讀者都容易辨識。
- `series=[0, 150, 1]` 將色票固定在 0–150 km；`background=True` 讓超過 150 km 的地震沿用最深端顏色，不會刪除這些事件；色條兩端的三角形表示超出範圍仍有對應顏色。`reverse=True` 可以反轉顏色順序。

**試看看**：只改 `cmap`，重跑下格，比較哪些深度比較醒目。深度要對照 color bar，不能只憑明暗判斷；目前圓圈有 40% 透明度，顏色也會受底圖和重疊影響。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [GMT 色票總覽](https://docs.generic-mapping-tools.org/6.5/reference/cpts.html) | 比較內建 CPT（色票圖鑑，非函式） | 色票名稱，如 `gmt/seis`、`jet` |
| [pygmt.makecpt()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.makecpt.html) | 建立數值到顏色的對應 | `cmap`、`series`、`reverse` |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 依地震深度上色 | `fill`、`cmap=True` |
| [fig.colorbar()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.colorbar.html) | 顯示色條與單位 | `frame` |


In [ ]:
import pygmt

from io import StringIO


# 沿用上一格的 magnitude_size 與 plot_quakes

# 1. 建立新圖與深度色票
# 深度色票固定為 0–150 km，方便不同圖之間比較

fig = pygmt.Figure()

# batlow：循序色票；也可換回 gmt/seis 比較可讀性
pygmt.makecpt(
    cmap="batlow",  # 新增：選擇色票名稱
    series=[0, 150, 1],  # 深度起點、終點、間隔（km）；上限固定 150
    background=True,  # 超過 150 km 沿用色票最深端的顏色
    continuous=True,  # 新增：建立連續漸層
)


# 2. 畫底圖與海岸線
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")
fig.basemap(map_scale="jBR+c24+w50k+o0.5c/0.5c+f+lkm")  # 50 km 比例尺，參考緯度 24°N


# 用深度控制填色，規模繼續控制大小
fig.plot(
    x=plot_quakes.longitude,
    y=plot_quakes.latitude,
    size=plot_quakes.mag.apply(magnitude_size),  # 沿用：規模仍控制大小
    style="c",
    fill=plot_quakes.depth,  # 改動：不再填固定色名，改傳入每筆深度
    cmap=True,  # 新增：用剛建立的 CPT 把深度數值轉成顏色
    pen="0.25p,gray20",
    transparency=40,  # 0 不透明，100 完全透明
)


# 3. 加上規模圖例（留出大圓需要的間距）
legend = StringIO(
    "".join(
        f"S 0.6c c {magnitude_size(mag):.3f}c gray70 0.25p,gray20 1.4c {label}\n"
        f"G {max(0.15, magnitude_size(mag) - 0.25):.2f}c\n"
        for mag, label in [
            (2, "M < 3"),
            (3, "3 <= M < 4"),
            (4, "4 <= M < 5"),
            (5, "5 <= M < 6"),
            (6, "6 <= M < 7"),
            (7, "M >= 7"),
        ]
    )
)

fig.legend(
    spec=legend,
    position="JTL+jTL+o0.2c",
    box="+gwhite+p0.5p",
)


# 4. 加上深度色條
fig.colorbar(
    position="JBC+w10c/0.35c+h+e",  # 三角形標示色階外的顏色
    frame=["xaf", "y+lDepth (km)"],  # 新增：顯示色票對應的刻度與深度單位
)


# 5. 顯示圖片
fig.show()


### 三張地震圖的圖說

USGS 目錄，2020-01-01 至執行當下（UTC），119–123°E、21–26°N、M ≥ 2，排除缺少位置、規模或深度的事件。第一張用等大圓點表示事件、星形標記 2024 花蓮主震；第二張以圓圈大小分級呈現規模；第三張再以顏色表示深度，超過 150 km 沿用端點色。這是多年地震分布，不是單一主震的餘震序列；USGS 對台灣小震收錄不完整。

修改查詢條件後，請同步修改標題與圖說，並記下實際下載日期。


## 資料來源與版本

- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)
- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)


| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 本課使用的真實事件資料 | 查詢條件包含在下載網址中 |

